In [1]:
import pandas as pd

ruta = "/workspaces/ANALISIS-DE-DATOS-UQ/Motor de simulacion salarial/global_ai_jobs_dataset.csv"
df = pd.read_csv(ruta)

df

,job_id,job_title,industry,company_size,country,city,experience_level,remote_type,education_required,ai_domain,salary_usd,years_experience_required,benefits_score,posting_date,skills,tools_used
0,1,AI Researcher,Manufacturing,SME,Canada,Berlin,Senior,Remote,PhD,Generative AI,104015,4,1.98,2021-04-01,Python;Computer Vision;SQL;NLP,Kubernetes;PyTorch;AWS
1,2,MLOps Engineer,Education,Startup,India,Tokyo,Entry,Remote,Bachelor,Recommendation Systems,103263,6,3.67,2020-04-12,NLP;PyTorch;Data Analysis;Computer Vision,Python;Spark;Docker
2,3,Data Analyst,E-commerce,SME,United Kingdom,Bangalore,Mid,Hybrid,Bachelor,Robotics,104190,7,2.26,2023-01-31,NLP;Computer Vision;Python;PyTorch,Python;Kubernetes;PyTorch
3,4,NLP Engineer,Technology,Enterprise,Brazil,Amsterdam,Mid,Remote,PhD,Recommendation Systems,51440,9,2.26,2022-09-30,Data Analysis;Statistics;TensorFlow;Python,Python;PyTorch;TensorFlow
4,5,AI Researcher,E-commerce,SME,Netherlands,Amsterdam,Senior,Hybrid,Master,Computer Vision,58358,0,2.44,2022-07-03,NLP;Machine Learning;PyTorch;SQL,TensorFlow;SQL;Azure
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119995,119996,Data Scientist,Telecommunications,Startup,Canada,Toronto,Lead,Remote,Bachelor,Recommendation Systems,165391,5,3.43,2021-06-13,Machine Learning;Deep Learning;Python;Computer...,SQL;Azure;Kubernetes
119996,119997,Prompt Engineer,Finance,Enterprise,India,Sydney,Senior,Onsite,Master,Recommendation Systems,114240,3,3.28,2022-02-27,Data Analysis;Deep Learning;Computer Vision;St...,GCP;TensorFlow;Docker
119997,119998,AI Product Manager,Education,SME,Australia,Tokyo,Entry,Hybrid,Bachelor,NLP,66159,3,1.72,2024-09-30,Machine Learning;Python;Computer Vision;Data A...,GCP;Spark;Python
119998,119999,Machine Learning Engineer,Manufacturing,SME,Brazil,London,Entry,Onsite,Bachelor,Generative AI,86297,5,1.71,2024-03-18,PyTorch;SQL;Machine Learning;Deep Learning,Docker;SQL;Azure


In [5]:
df.describe()


,job_id,salary_usd,years_experience_required,benefits_score
count,120000.000000,120000.000000,120000.000000,120000.000000
mean,60000.500000,119995.002358,4.487683,2.995962
std,34641.160489,46150.388458,2.874546,1.153465
min,1.000000,40001.000000,0.000000,1.000000
25%,30000.750000,80165.750000,2.000000,2.000000
50%,60000.500000,119914.500000,4.000000,2.990000
75%,90000.250000,159840.000000,7.000000,3.990000
max,120000.000000,199999.000000,9.000000,5.000000


In [2]:
# Identificar variables categóricas
categorical_cols = df.select_dtypes(include=['object']).columns
print("Variables categóricas:", list(categorical_cols))

# Calcular la moda para cada variable categórica
modes = {}
for col in categorical_cols:
    mode_values = df[col].mode()
    modes[col] = mode_values.tolist() if not mode_values.empty else None

# Mostrar las modas
for col, mode in modes.items():
    print(f"Moda de {col}: {mode}")

/tmp/ipykernel_20045/4266739755.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include=['object']).columns


Variables categóricas: ['job_title', 'industry', 'company_size', 'country', 'city', 'experience_level', 'remote_type', 'education_required', 'ai_domain', 'posting_date', 'skills', 'tools_used']
Moda de job_title: ['Prompt Engineer']
Moda de industry: ['Finance']
Moda de company_size: ['Enterprise']
Moda de country: ['Germany']
Moda de city: ['London']
Moda de experience_level: ['Mid']
Moda de remote_type: ['Onsite']
Moda de education_required: ['Bachelor']
Moda de ai_domain: ['Computer Vision']
Moda de posting_date: ['2025-11-02']
Moda de skills: ['SQL;Machine Learning;PyTorch;Deep Learning', 'SQL;TensorFlow;Machine Learning;Deep Learning']
Moda de tools_used: ['Python;Docker;Azure']


# Problemas del Conjunto de datos

El conjunto de datos presenta varias limitaciones que deben ser consideradas antes de desarrollar el modelo de regresión y el posterior motor de simulación. En primer lugar, se observa una alta predominancia de variables categóricas, como el cargo, la industria, el país o la ciudad, lo que implica la necesidad de aplicar técnicas de codificación (como variables dummy). Este proceso puede incrementar significativamente la dimensionalidad del modelo, generando riesgos de sobreajuste y reduciendo su capacidad de generalización.

Adicionalmente, la variable de habilidades (skills) se encuentra almacenada como texto no estructurado, lo que dificulta su uso directo en el modelo. Para incorporarla, es necesario transformarla en múltiples variables binarias, lo cual no solo incrementa la complejidad del análisis, sino que también puede introducir problemas de multicolinealidad, dado que muchas habilidades tienden a estar altamente correlacionadas entre sí.

Por otra parte, se identifican posibles inconsistencias internas en los datos, como combinaciones poco coherentes entre nivel de experiencia, años requeridos y salario. Este tipo de ruido puede afectar la estimación de los coeficientes del modelo y, en consecuencia, distorsionar los resultados del ejercicio de simulación.

Asimismo, el dataset presenta un problema de variables omitidas, ya que no incluye factores relevantes como habilidades blandas, calidad de la formación, redes de contacto o características específicas de las empresas. La ausencia de estas variables puede generar sesgos en las estimaciones, limitando la capacidad explicativa del modelo.

Otro aspecto importante es la posible naturaleza artificial o simplificada de los datos, evidenciada en la distribución relativamente uniforme de los salarios y la ausencia de valores extremos. Esto sugiere que el conjunto de datos podría no reflejar completamente la complejidad del mercado laboral real, lo que afectaría la validez externa de los resultados.

Finalmente, aunque se dispone de una variable temporal (fecha de publicación), esta no se encuentra estructurada para su análisis directo, lo que implica una subutilización de la dimensión temporal. No incorporar esta información limita la posibilidad de capturar tendencias en la evolución de los salarios a lo largo del tiempo.



